# Machine translation

In [ ]:
import pandas as pd
import requests
import time
from tqdm.auto import tqdm

import os
HF_TOKEN = os.getenv("HF_TOKEN")

MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"
API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json"
}


## Translation function
This section defines the function used to translate Arabic text to English using the Hugging Face chat completion endpoint.

In [31]:
def translate_text(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a professional Arabic-to-English translator. "
                    "Translate the given Arabic text into fluent English. "
                    "Return ONLY the English translation without explanations."
                )
            },
            {
                "role": "user",
                "content": str(text)
            }
        ],
        "temperature": 0,
        "max_tokens": 512,
    }

    try:
        response = requests.post(
            API_URL,
            headers=headers,
            json=payload,
            timeout=120,
        )
        response.raise_for_status()
        result = response.json()

        if isinstance(result, dict) and result.get("error"):
            raise Exception(result["error"])

        if isinstance(result, dict) and "choices" in result and len(result["choices"]) > 0:
            message = result["choices"][0].get("message", {})
            content = message.get("content")
            if content:
                return content.strip()

        return str(result).strip()

    except requests.exceptions.RequestException as e:
        print(f"Translation Error: {e}")
        if hasattr(e, 'args') and any('NameResolutionError' in str(arg) for arg in e.args):
            print("Network error: cannot resolve Hugging Face host. Check your internet connection or DNS settings.")
        return ""
    except Exception as e:
        print(f"Translation Error: {e}")
        return ""


    except requests.exceptions.RequestException as e:        print(f"Translation Error: {e}")

## Load dataset
Load the original Arabic dataset so it can be translated and saved.

In [32]:
df = pd.read_csv("AAFAQ_Dataset.csv")[:5]
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (5, 3)


,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


## Translate questions
Translate the Arabic questions into English using the translation API.

In [33]:
# Quick API sanity check
print(translate_text('مرحبا'))

Hello


In [34]:
tqdm.pandas()

df["QuestionText_EN"] = df["QuestionText"].progress_apply(
    translate_text
)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
df['QuestionText_EN']

0    Which is better, studying in the past or in th...
1        Isn't cotton the backbone of wealth in Egypt?
2                       Does the sun rise in the east?
3    Do you know that bacteria are microscopic livi...
4               Is air primarily composed of nitrogen?
Name: QuestionText_EN, dtype: object

## Translate answers
Translate the Arabic answers into English with the same translation function.

In [36]:
df["Answer_EN"] = df["Answer"].progress_apply(
    translate_text
)

  0%|          | 0/5 [00:00<?, ?it/s]

## Preview results
Inspect a sample of the translated questions and answers before saving.

In [39]:
df[
    [
        "QuestionText",
        "QuestionText_EN",
        "Answer",
        "Answer_EN"
    ]
].head()

,QuestionText,QuestionText_EN,Answer,Answer_EN
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,"Which is better, studying in the past or in th...",الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,Studying nowadays is considered better due to ...
1,أليس القطن عماد الثروة في مصر؟,Isn't cotton the backbone of wealth in Egypt?,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,Cotton is considered one of the most important...
2,أتصعد الشمس من الشرق؟,Does the sun rise in the east?,الشمس تصعد من الشرق.,The sun rises in the east.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,Do you know that bacteria are microscopic livi...,البكتيريا تُعرف بأنها كائنات حية دقيقة.,Bacteria are known as microorganisms.
4,أيتكون الهواء أساساً من النيتروجين؟,Is air primarily composed of nitrogen?,الهواء يتكون أساساً من النيتروجين.,Air is composed primarily of nitrogen.


## Save translated dataset
Write the augmented dataset to a new CSV file with translations included.

In [38]:
output_file = "AAFAQ_Dataset_Translated.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nTranslated dataset saved as: {output_file}")


Translated dataset saved as: AAFAQ_Dataset_Translated.csv
